In [ ]:
!pip install -r requirements_whisper

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements_whisper.json'

[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
import re
import unicodedata
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from jiwer import wer

PhoASR - Matches audio context files in speech_data with corresponding id field in transcription.json. Done by reading 2 json files and match the closest

In [27]:
# ============================================================
# Paths
# ============================================================

TRANSCRIPTION_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\rq1\transcription_phoasr_train.json")
GROUNDTRUTH_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\rq1\uit_viquad_train_squad.json")
OUTPUT_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\rq1\context_mapping_phoasr_VIQUAD_train.json")
REJECTED_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\rq1\rejected_phoasr_viquad_train.json")
TOP_K = 5

In [28]:
# Maximum acceptable WER for a match.
#
# Example:
#   0.30 = 30% WER
#
# Increase this if too many legitimate matches are rejected.
# Decrease this if unrelated audio files are being matched.
MAX_WER = 2

# Number of candidates retrieved by TF-IDF before WER
# verification.
TOP_K = 5


# ============================================================
# Text normalization
# ============================================================

def normalize_text(text):
    """
    Normalize text before matching.

    Vietnamese diacritics are preserved.
    """

    text = unicodedata.normalize("NFC", text)
    text = text.lower()

    # Remove punctuation
    text = re.sub(
        r"[^\w\sÀ-ỹ]",
        " ",
        text,
        flags=re.UNICODE
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ============================================================
# Extract context ID prefix
# ============================================================

def get_context_id_prefix(qa_id):

    parts = qa_id.split("-")

    if len(parts) < 2:
        return None

    return f"{parts[0]}-{parts[1]}"

In [31]:

# ============================================================
# Load ASR transcriptions
# ============================================================

with open(TRANSCRIPTION_FILE, "r", encoding="utf-8") as f:
    transcriptions = json.load(f)


# ============================================================
# Load ViSQA JSON
# ============================================================

with open(GROUNDTRUTH_FILE, "r", encoding="utf-8") as f:
    groundtruth = json.load(f)


# ============================================================
# Extract contexts
# ============================================================

contexts = []

for data_item in groundtruth["data"]:

    for paragraph in data_item["paragraphs"]:

        context = paragraph["context"]
        qas = paragraph["qas"]

        if not context or not qas:
            continue

        # Get context ID from the first QA.
        #
        # Example:
        # 0057-0077-0001
        #       ↓
        # 0057-0077
        context_id = get_context_id_prefix(
            qas[0]["id"]
        )

        contexts.append({
            "context_id": context_id,
            "context": context,
            "qas": qas
        })


print(f"Loaded {len(contexts)} unique contexts")
print(f"Loaded {len(transcriptions)} audio transcriptions")


# ============================================================
# Normalize contexts
# ============================================================

normalized_contexts = [
    normalize_text(item["context"])
    for item in contexts
]


# ============================================================
# Character n-gram TF-IDF
# ============================================================

vectorizer = TfidfVectorizer(
    analyzer="char",

    # 3-5 character n-grams
    ngram_range=(3, 5),

    min_df=1,

    # Helps reduce the effect of very frequent n-grams
    sublinear_tf=True
)

context_vectors = vectorizer.fit_transform(
    normalized_contexts
)


# ============================================================
# Storage for results
# ============================================================

results = []
rejected_files = []


# ============================================================
# Match every audio transcription
# ============================================================

for i, item in enumerate(transcriptions):

    audio_file = item["file"]
    transcription = item["transcription"]

    normalized_transcription = normalize_text(
        transcription
    )

    # --------------------------------------------------------
    # Handle empty transcription
    # --------------------------------------------------------

    if not normalized_transcription:

        rejected_entry = {
            "audio_file": audio_file,
            "transcription": transcription,
            "reason": "empty_transcription"
        }

        rejected_files.append(rejected_entry)

        continue


    # --------------------------------------------------------
    # Convert ASR transcription to TF-IDF vector
    # --------------------------------------------------------

    transcription_vector = vectorizer.transform(
        [normalized_transcription]
    )


    # --------------------------------------------------------
    # Calculate cosine similarity against all contexts
    # --------------------------------------------------------

    similarities = cosine_similarity(
        transcription_vector,
        context_vectors
    )[0]


    # --------------------------------------------------------
    # Retrieve top-k candidates
    # --------------------------------------------------------

    top_indices = similarities.argsort()[-TOP_K:][::-1]


    candidates = []


    # --------------------------------------------------------
    # Calculate WER for candidates
    # --------------------------------------------------------

    for idx in top_indices:

        candidate_context = normalized_contexts[idx]

        similarity = similarities[idx]

        candidate_wer = wer(
            candidate_context,
            normalized_transcription
        )

        candidates.append({
            "context_id": contexts[idx]["context_id"],
            "similarity": float(similarity),
            "wer": float(candidate_wer)
        })


    # --------------------------------------------------------
    # Select candidate with lowest WER
    # --------------------------------------------------------

    best_candidate = min(
        candidates,
        key=lambda x: x["wer"]
    )

    best_context_id = best_candidate["context_id"]

    best_context = contexts[
        next(
            idx
            for idx in top_indices
            if contexts[idx]["context_id"] == best_context_id
        )
    ]
    # --------------------------------------------------------
    # Apply WER cutoff
    # --------------------------------------------------------

    if best_candidate["wer"] > MAX_WER:

        rejected_entry = {
            "audio_file": audio_file,
            "transcription": transcription,
            "matched_context": best_context["context"],
            "best_context_id": best_context_id,

            "best_similarity": (
                best_candidate["similarity"]
            ),

            "best_wer": best_candidate["wer"],

            "wer_cutoff": MAX_WER,

            "reason": "best_candidate_exceeds_wer_cutoff",

            "candidates": candidates
        }

        rejected_files.append(rejected_entry)

    else:

        # ----------------------------------------------------
        # Find matched context
        # ----------------------------------------------------

        best_context = next(
            context
            for context in contexts
            if context["context_id"] == best_context_id
        )


        # ----------------------------------------------------
        # Store accepted result
        # ----------------------------------------------------

        results.append({
            "audio_file": audio_file,

            "transcription": transcription,

            "context_id": best_context_id,

            "matched_context": best_context["context"],

            "qas": best_context["qas"],

            "similarity": best_candidate["similarity"],

            "wer": best_candidate["wer"],

            "candidates": candidates
        })


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (i + 1) % 10 == 0:

        print(
            f"Processed "
            f"{i + 1}/{len(transcriptions)}"
        )



# ============================================================
# Build summary
# ============================================================

summary = {
    "transcription_files": len(transcriptions),
    "groundtruth_contexts": len(contexts),
    "accepted_results": len(results),
    "missing_transcriptions": len(rejected_files),
    "rejected_files": len(rejected_files)
}

# ============================================================
# Save accepted matches
# ============================================================

output = {
    "summary": summary,
    "results": results
}


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Save rejected files
# ============================================================

with open(REJECTED_FILE, "w", encoding="utf-8") as f:

    json.dump(
        rejected_files,
        f,
        ensure_ascii=False,
        indent=2)

Loaded 4101 unique contexts
Loaded 4101 audio transcriptions
Processed 10/4101
Processed 20/4101
Processed 30/4101
Processed 40/4101
Processed 50/4101
Processed 60/4101
Processed 70/4101
Processed 80/4101
Processed 90/4101
Processed 100/4101
Processed 110/4101
Processed 120/4101
Processed 130/4101
Processed 140/4101
Processed 150/4101
Processed 160/4101
Processed 170/4101
Processed 180/4101
Processed 190/4101
Processed 200/4101
Processed 210/4101
Processed 220/4101
Processed 230/4101
Processed 240/4101
Processed 250/4101
Processed 260/4101
Processed 270/4101
Processed 280/4101
Processed 290/4101
Processed 300/4101
Processed 310/4101
Processed 320/4101
Processed 330/4101
Processed 340/4101
Processed 350/4101
Processed 360/4101
Processed 370/4101
Processed 380/4101
Processed 390/4101
Processed 400/4101
Processed 410/4101
Processed 420/4101
Processed 430/4101
Processed 440/4101
Processed 450/4101
Processed 460/4101
Processed 470/4101
Processed 480/4101
Processed 490/4101
Processed 500/410

In [32]:
# ============================================================
# Summary
# ============================================================

total = len(transcriptions)

accepted = len(results)

rejected = len(rejected_files)

acceptance_rate = (
    accepted / total * 100
    if total > 0
    else 0
)

rejection_rate = (
    rejected / total * 100
    if total > 0
    else 0
)


print()
print("=" * 60)
print("MATCHING COMPLETE")
print("=" * 60)

print(f"Total audio files:     {total}")
print(f"Accepted matches:      {accepted}")
print(f"Rejected files:        {rejected}")

print(
    f"Acceptance rate:       "
    f"{acceptance_rate:.2f}%"
)

print(
    f"Rejection rate:        "
    f"{rejection_rate:.2f}%"
)

print()
print(f"WER cutoff:             {MAX_WER:.2f}")
print()
print(f"Accepted output:        {OUTPUT_FILE}")
print(f"Rejected output:        {REJECTED_FILE}")
print("=" * 60)


MATCHING COMPLETE
Total audio files:     4101
Accepted matches:      4100
Rejected files:        1
Acceptance rate:       99.98%
Rejection rate:        0.02%

WER cutoff:             2.00

Accepted output:        G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\rq1\context_mapping_phoasr_VIQUAD_train.json
Rejected output:        G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\rq1\rejected_phoasr_viquad_train.json


ViSQA-UiT-ViQUAD matching - since both files have the same structure, match by id

In [2]:
# ============================================================
# Paths
# ============================================================

TRANSCRIPTION_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\rq1\transcription_visqa_val.json")
GROUNDTRUTH_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\rq1\uit_viquad_val.json")
OUTPUT_FILE = Path(r"G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\rq2\input\context_mapping_visqa_VIQUAD_val.json")

In [19]:
# ============================================================
# Load JSON files
# ============================================================

with open(TRANSCRIPTION_FILE, "r", encoding="utf-8") as f:
    transcriptions = json.load(f)

with open(GROUNDTRUTH_FILE, "r", encoding="utf-8") as f:
    groundtruth = json.load(f)


# ============================================================
# Flatten SQuAD structure
# ============================================================

def extract_squad_entries(squad_data):
    entries = []

    for article in squad_data["data"]:

        for paragraph in article["paragraphs"]:

            context = paragraph["context"]

            for qa in paragraph["qas"]:

                entries.append({
                    "id": qa["id"],
                    "context": context,
                    "qa": qa
                })

    return entries


transcription_entries = extract_squad_entries(transcriptions)
groundtruth_entries = extract_squad_entries(groundtruth)

# ============================================================
# Helper: Extract context_id
# ============================================================

def get_context_id(qa_id):
    # 0001-0001-0001 -> 0001-0001
    return "-".join(qa_id.split("-")[:2])


# ============================================================
# Index groundtruth by context_id
# ============================================================

groundtruth_index = {}

for item in groundtruth_entries:

    context_id = get_context_id(item["id"])

    if context_id not in groundtruth_index:
        groundtruth_index[context_id] = {
            "context": item["context"],
            "qas": []
        }

    groundtruth_index[context_id]["qas"].append(
        item["qa"]
    )


# ============================================================
# Match transcription contexts with groundtruth contexts
# ============================================================

results = []
unmatched = []

processed_context_ids = set()


for item in transcription_entries:

    context_id = get_context_id(item["id"])


    # Match each context only once
    if context_id in processed_context_ids:
        continue

    processed_context_ids.add(context_id)


    if context_id not in groundtruth_index:
        unmatched.append(context_id)
        continue


    gt_item = groundtruth_index[context_id]


    results.append({

        "context_id": context_id,

        # Transcription context
        "transcription": item["context"],

        # Groundtruth context
        "matched_context": gt_item["context"],

        # Groundtruth QA entries
        "qas": gt_item["qas"]

    })

# ============================================================
# Count unique contexts
# ============================================================

transcription_contexts = set(
    item["context"]
    for item in transcription_entries
)

groundtruth_contexts = set(
    item["context"]
    for item in groundtruth_entries
)


print("\n===== Dataset Summary =====")

print(
    f"Transcript QA entries: {len(transcription_entries)}"
)

print(
    f"Transcript contexts: {len(transcription_contexts)}"
)

print(
    f"Groundtruth QA entries: {len(groundtruth_entries)}"
)

print(
    f"Groundtruth contexts: {len(groundtruth_contexts)}"
)

print(
    f"Matched results: {len(results)}"
)

print(
    f"Unmatched transcription IDs: {len(unmatched)}"
)

print("===========================\n")

# ============================================================
# Build summary
# ============================================================

summary = {
    "transcription_contexts": len(transcription_contexts),
    "groundtruth_contexts": len(groundtruth_contexts),
    "accepted_results": len(results),
    "missing_transcriptions": len(unmatched),
    "rejected_files": 0
}


# ============================================================
# Save output
# ============================================================

output = {
    "summary": summary,
    "results": results
}

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"Saved output: {OUTPUT_FILE}"
)

print(
    f"Matched contexts: {len(results)}"
)

print(
    f"Missing groundtruth context IDs: {len(unmatched)}"
)




===== Dataset Summary =====
Transcript QA entries: 1834
Transcript contexts: 550
Groundtruth QA entries: 3814
Groundtruth contexts: 557
Matched results: 550
Unmatched transcription IDs: 0

Saved output: G:\Uni\Sem2\NLP\term_paper\1858385_NLP_Term_paper_SoSE2025-master\data\rq2\input\context_mapping_visqa_VIQUAD_val.json
Matched contexts: 550
Missing groundtruth context IDs: 0
